In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
bronze_df = spark.table("load_json.bronze.raw_json_documents")

display(bronze_df)

bronze_df.printSchema()

In [0]:
orders_df = bronze_df.select(explode(col("records")).alias("orders"))

display(orders_df)
orders_df.printSchema()

In [0]:
silver_df =orders_df.select(
    col("orders.order_id").alias("order_id"),
    col("orders.order_date").alias("order_date"),
    col("orders.customer.customer_id").alias("customer_id"),
    col("orders.customer.name").alias("customer_name"),
    col("orders.customer.email").alias("email"),
    col("orders.customer.phone").alias("phone"),
    col("orders.customer.address.city").alias("city"),
    col("orders.customer.address.state").alias("state"),
    col("orders.customer.address.pincode").alias("pincode"),
    col("orders.payment.method").alias("payment_method"),
    col("orders.payment.status").alias("payment_status"),
    col("orders.payment.amount").alias("payment_amount"),
    col("orders.payment.currency").alias("currency"),

    col("orders.shipping.carrier").alias("shipping_carrier"),
    col("orders.shipping.tracking_id").alias("tracking_id"),
    col("orders.shipping.delivered").alias("delivered"),

    col("orders.order_status").alias("order_status"),

    col("orders.items").alias("items"),
    col("orders.discount_codes").alias("discount_codes")


)

display(silver_df)

In [0]:


order_items_df = silver_df.select(
    col("order_id"),
    explode_outer(col("items")).alias("items")
)
display(order_items_df)


In [0]:
order_items_df=order_items_df.select(

    col("order_id").alias("order_id"),
    col("items.product_id").alias("product_id"),
    col("items.name").alias("product_name"),
    col("items.category").alias("category"),
    col("items.quantity").alias("quantity"),
    col("items.unit_price").alias("unit_price"),
    col("items.tags").alias("tags")

)
display(order_items_df)

In [0]:
silver_df =( 
            
silver_df
.withColumn("customer_name", trim(col("customer_name"))
).withColumn("city",trim(col("city"))
             
).withColumn("email",lower(trim(col("email")))
                           
).withColumn("state",trim(col("state"))
).withColumn("currency",upper(trim(col("currency")))
).withColumn("payment_method",upper(trim(col("payment_method")))
).withColumn("shipping_carrier",upper(trim(col("shipping_carrier")))
).withColumn("order_status",
            when(col("order_status")=="processing","Processing")
            .when(col("order_status")=="shipped","Shipped")
            .when(col("order_status")=="delivered","Delivered")
            .when(col("order_status")=="cancelled","Cancelled")
            .otherwise(trim(col("order_status")))
).withColumn("payment_status",
            when(col("payment_status")=="paid","Paid")
            .when(col("payment_status")=="unpaid","Unpaid")
            .when(col("payment_status")=="refunded","Refunded")
            .otherwise(trim(col("payment_status")))
).withColumn(
        "order_date",
        when(
            col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"),
            to_date(col("order_date"), "yyyy-MM-dd")
        ).when(
            col("order_date").rlike(r"^\d{4}/\d{2}/\d{2}$"),
            to_date(col("order_date"), "yyyy/MM/dd")
        ).otherwise(None)
).withColumn(
        "customer_name",
        when(col("customer_name") == "", None)
        .otherwise(col("customer_name"))
).withColumn(
        "email",
        when(col("email") == "", None).otherwise(col("email"))
).withColumn(
        "city",
        when(col("city") == "", None).otherwise(col("city"))
)
)

display(silver_df)



            

In [0]:
valid_order_df = silver_df.filter(

    col("order_id").isNotNull() &
    col("order_date").isNotNull() &
    col("customer_id").isNotNull() &
    col("customer_name").isNotNull() 
 
).drop("items", "discount_codes")


quarantine_orders_df = silver_df.filter(
    col("order_id").isNull() |
    col("order_date").isNull() |
    col("customer_id").isNull() |
    col("customer_name").isNull() 
   
).drop("items", "discount_codes")

    

In [0]:
print("Valid orders:", valid_order_df.count())
print("Quarantined orders:", quarantine_orders_df.count())

display(valid_order_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS load_json.silver")

In [0]:
(
    valid_order_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("load_json.silver.orders")


)

In [0]:
silver_valid_table = spark.table("load_json.silver.orders")


In [0]:

display(order_items_df)

In [0]:

order_items_df=(

    order_items_df.withColumn("product_name",trim(col("product_name")))
    .withColumn("category",upper(trim(col("category"))))

)



valid_oi_df = order_items_df.filter(
    col("order_id").isNotNull() &
    col("product_id").isNotNull() & 
    col("product_name").isNotNull() 
   
)

quarantine_oi_df = order_items_df.filter(
    col("order_id").isNull() |
    col("product_id").isNull() |
    col("product_name").isNull() 
   
)
print("Valid order items:", valid_oi_df.count())
print("Quarantined order items:", quarantine_oi_df.count())

display(valid_oi_df)

In [0]:
(

    valid_oi_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("load_json.silver.order_items")
)

order_items_silver = spark.table("load_json.silver.order_items")

display(order_items_silver)